# HR Employee Attrition: Analytics & Model Training (8 Inputs)

Run this notebook to inspect and clean the dataset, explore patterns, train a Logistic Regression model using **only 8 prediction inputs**, evaluate it, and save `attrition_model.sav`.

Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` beside this notebook before running.

In [ ]:
# If needed, install: %pip install pandas numpy matplotlib seaborn scikit-learn joblib openpyxl
import os, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef, log_loss,
    ConfusionMatrixDisplay, RocCurveDisplay)
warnings.filterwarnings("ignore")
DATA_PATH="WA_Fn-UseC_-HR-Employee-Attrition.csv"
OUTPUT_DIR="outputs"
os.makedirs(OUTPUT_DIR,exist_ok=True)
TARGET="Attrition"
FEATURES=["Age","BusinessTravel","Department","JobRole","MaritalStatus","MonthlyIncome","OverTime","YearsAtCompany"]
RANDOM_STATE=42


## 1. Load and inspect

In [ ]:
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
display(df.head())
display(df.describe(include="all").T)


## 2. Check missing values, duplicates, and clean

In [ ]:
missing_before=df.isna().sum()
duplicates=int(df.duplicated().sum())
clean=df.drop_duplicates().copy()
for col in clean.columns:
    if clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(clean[col]):
            clean[col]=clean[col].fillna(clean[col].median())
        else:
            modes=clean[col].mode(dropna=True)
            clean[col]=clean[col].fillna(modes.iloc[0] if len(modes) else "Unknown")
print("Duplicate rows removed:",duplicates)
print("Missing cells before:",int(missing_before.sum()))
print("Missing cells after:",int(clean.isna().sum().sum()))
display(pd.DataFrame({"Missing Before":missing_before,"Missing After":clean.isna().sum()}))
clean.to_csv(os.path.join(OUTPUT_DIR,"cleaned_hr_employee_attrition.csv"),index=False)


## 3. Exploratory Data Analysis

In [ ]:
display(clean[TARGET].value_counts().rename_axis("Attrition").to_frame("Count"))
plt.figure(figsize=(6,4))
sns.countplot(data=clean,x=TARGET)
plt.title("Attrition Distribution"); plt.tight_layout(); plt.show()

display(clean.select_dtypes(include=np.number).describe().T)
clean.select_dtypes(include=np.number).hist(figsize=(16,12),bins=20)
plt.tight_layout(); plt.show()


In [ ]:
# Attrition rates for selected categories
for col in ["BusinessTravel","Department","JobRole","MaritalStatus","OverTime"]:
    rates=clean.assign(_attr=clean[TARGET].astype(str).str.lower().eq("yes")).groupby(col)["_attr"].mean().sort_values(ascending=False)*100
    display(rates.rename("Attrition Rate (%)").to_frame())
    rates.plot(kind="bar",figsize=(8,4),title=f"Attrition Rate by {col}",ylabel="Attrition rate (%)")
    plt.tight_layout(); plt.show()


## 4. Select eight model inputs

Inputs: Age, Business Travel, Department, Job Role, Marital Status, Monthly Income, Overtime, and Years at Company. Target: Attrition (Yes=1, No=0). The selection keeps the Streamlit form short; it is a practical feature subset, not a claim that these are the only drivers.

In [ ]:
missing_features=[c for c in FEATURES+[TARGET] if c not in clean.columns]
if missing_features: raise ValueError(f"Missing expected columns: {missing_features}")
X=clean[FEATURES].copy()
y=clean[TARGET].astype(str).str.strip().str.lower().map({"yes":1,"no":0})
if y.isna().any(): raise ValueError("Attrition target should contain Yes/No.")
display(pd.DataFrame({"Selected Feature":FEATURES,"Type":[str(X[c].dtype) for c in FEATURES]}))


## 5. 80/20 split and model training

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
numeric=[c for c in FEATURES if pd.api.types.is_numeric_dtype(X[c])]
categorical=[c for c in FEATURES if c not in numeric]
preprocessor=ColumnTransformer([
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numeric),
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical)
])
model=Pipeline([("preprocessor",preprocessor),
    ("classifier",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=RANDOM_STATE))])
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
y_prob=model.predict_proba(X_test)[:,1]
print("Train:",len(X_train),"Test:",len(X_test))


## 6. Evaluation

In [ ]:
tn,fp,fn,tp=confusion_matrix(y_test,y_pred,labels=[0,1]).ravel()
metrics={
 "Accuracy":accuracy_score(y_test,y_pred),
 "Balanced Accuracy":balanced_accuracy_score(y_test,y_pred),
 "Precision (Attrition=Yes)":precision_score(y_test,y_pred,zero_division=0),
 "Recall (Attrition=Yes)":recall_score(y_test,y_pred,zero_division=0),
 "F1 Score (Attrition=Yes)":f1_score(y_test,y_pred,zero_division=0),
 "ROC-AUC":roc_auc_score(y_test,y_prob),
 "MCC":matthews_corrcoef(y_test,y_pred),"Log Loss":log_loss(y_test,y_prob),
 "True Negative":int(tn),"False Positive":int(fp),"False Negative":int(fn),"True Positive":int(tp)
}
metrics_df=pd.DataFrame(metrics.items(),columns=["Metric","Value"])
display(metrics_df)
print(classification_report(y_test,y_pred,target_names=["No Attrition","Attrition"],zero_division=0))
cm=pd.DataFrame([[tn,fp],[fn,tp]],index=["Actual No","Actual Yes"],columns=["Predicted No","Predicted Yes"])
display(cm)
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=["No","Yes"],cmap="Blues")
plt.tight_layout(); plt.show()
RocCurveDisplay.from_predictions(y_test,y_prob)
plt.tight_layout(); plt.show()
print("R² is not applicable to this binary classification task.")


## 7. Save cleaned data, reports, and trained model

In [ ]:
joblib.dump(model,os.path.join(OUTPUT_DIR,"attrition_model.sav"))
metrics_df.to_csv(os.path.join(OUTPUT_DIR,"model_metrics.csv"),index=False)
cm.to_csv(os.path.join(OUTPUT_DIR,"confusion_matrix.csv"))
pd.DataFrame(classification_report(y_test,y_pred,target_names=["No Attrition","Attrition"],output_dict=True,zero_division=0)).T.to_csv(
    os.path.join(OUTPUT_DIR,"classification_report.csv"))
pd.DataFrame({"Feature":FEATURES}).to_csv(os.path.join(OUTPUT_DIR,"model_features.csv"),index=False)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR,"analytics_summary.xlsx"),engine="openpyxl") as writer:
    metrics_df.to_excel(writer,sheet_name="Metrics",index=False)
    cm.to_excel(writer,sheet_name="Confusion Matrix")
    pd.DataFrame({"Column":df.columns,"Missing Before":[int(df[c].isna().sum()) for c in df.columns],
                  "Missing After":[int(clean[c].isna().sum()) for c in df.columns]}).to_excel(writer,sheet_name="Missing Values",index=False)
    clean.describe(include="all").T.to_excel(writer,sheet_name="Data Summary")
print("Saved model to:",os.path.abspath(os.path.join(OUTPUT_DIR,"attrition_model.sav")))
print("Saved outputs to:",os.path.abspath(OUTPUT_DIR))
